# Module 4: Deploy to Production

You've trained the model, optimized it, and exported it. Now let's ship it.

In this module, you'll:

1. Build a **FastAPI inference API** with proper error handling
2. **Containerize** it with Docker (production-grade Dockerfile)
3. **Deploy** to Google Cloud Run
4. Understand what to **monitor** in production

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/arj7192/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete - GPU:", os.environ.get("COLAB_GPU", "not detected"))

## 4.1 The Inference API

Let's walk through the FastAPI server at `serve/app.py`.

Key production patterns:
- **Model loaded once at startup** (not per-request)
- **Health endpoint** (`/health`) for load balancers
- **Input validation** with Pydantic
- **Structured error responses**
- **Request timing** middleware

In [ ]:
import sys
sys.path.insert(0, '..')

# Let's look at the serving code
from pathlib import Path

app_code = Path('../serve/app.py').read_text()
print(app_code)

### Anatomy of the API

```
POST /generate
{
    "prompt": "The meaning of life is",
    "max_tokens": 50,
    "temperature": 0.8
}
→
{
    "text": "The meaning of life is ...",
    "tokens_generated": 50,
    "latency_ms": 123.4
}
```

```
GET /health
→
{
    "status": "healthy",
    "model_loaded": true,
    "device": "cpu"
}
```

## 4.2 Run Locally

First, test the server locally before containerizing.

In [ ]:
# The server expects a checkpoint and tokenizer in the serve/ directory.
# If no checkpoint exists yet (fresh Colab session), we'll train one first.
import shutil
import torch

serve_dir = Path('../serve')

# Ensure tokenizer exists
tokenizer_src = Path('../tokenizer.json')
if not tokenizer_src.exists():
    from src.data import prepare_wikitext2
    print("Preparing tokenizer...")
    prepare_wikitext2(vocab_size=8192, seq_len=128, tokenizer_path=str(tokenizer_src))

# Ensure checkpoint exists - train if needed
from src.utils import CheckpointManager, set_seed, get_device
ckpt_manager = CheckpointManager('../checkpoints')
latest = ckpt_manager.latest()

if not latest:
    print("No checkpoint found - training a model (3 epochs with AMP)...")
    from src.model import build_model
    from src.data import prepare_wikitext2, create_dataloaders
    from src.evaluate import evaluate

    device = get_device()
    set_seed(42)
    train_ds, val_ds, _, tok = prepare_wikitext2(vocab_size=8192, seq_len=128, tokenizer_path=str(tokenizer_src))
    cfg = {'vocab_size': tok.get_vocab_size(), 'd_model': 256, 'n_heads': 4, 'd_ff': 512, 'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1}
    mdl = build_model(cfg).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), lr=3e-4)
    loader, val_loader = create_dataloaders(train_ds, val_ds, batch_size=64)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    for epoch in range(3):
        mdl.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
                loss = mdl(x, targets=y)['loss']
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                opt.step()
            opt.zero_grad(set_to_none=True)
        val_m = evaluate(mdl, val_loader, device, use_amp=use_amp)
        print(f"  Epoch {epoch+1}/3 | val_loss {val_m['val_loss']:.4f}")
    ckpt_manager.save(mdl, opt, 3, 0, val_m['val_loss'], cfg)
    latest = ckpt_manager.latest()
    print(f"  Checkpoint saved: {latest}")

# Copy artifacts to serve/
shutil.copy(tokenizer_src, serve_dir / 'tokenizer.json')
print(f"Copied tokenizer to {serve_dir / 'tokenizer.json'}")

shutil.copy(latest, serve_dir / 'model_checkpoint.pt')
print(f"Copied checkpoint to {serve_dir / 'model_checkpoint.pt'}")

### Start the server

**Locally** you'd run `cd serve && uvicorn app:app --host 0.0.0.0 --port 8000` in a terminal.

In **Colab** (no terminal), we start it in a background thread and test from the next cell.

In [ ]:
import subprocess, time, threading, requests

# Start uvicorn in a background process
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="../serve",
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# Stream server logs in a background thread so we can see startup
def _print_logs(proc):
    for line in proc.stderr:
        print(f"[server] {line.decode().strip()}")
log_thread = threading.Thread(target=_print_logs, args=(server_proc,), daemon=True)
log_thread.start()

# Wait for the server to be ready
print("Starting server...")
for _ in range(30):
    try:
        requests.get("http://localhost:8000/health", timeout=1)
        print("Server is up!\n")
        break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("Server failed to start - check logs above.")

In [ ]:
# --- Test the API ---

# Health check
resp = requests.get("http://localhost:8000/health")
print("=== Health Check ===")
print(resp.json())

# Generate text
print("\n=== Text Generation ===")
for prompt in ["The future of AI", "In the beginning", "Scientists recently"]:
    resp = requests.post(
        "http://localhost:8000/generate",
        json={"prompt": prompt, "max_tokens": 40, "temperature": 0.8},
    )
    result = resp.json()
    print(f"\nPrompt: {result['prompt']}")
    print(f"Output: {result['text'][:150]}")
    print(f"Latency: {result['latency_ms']:.1f} ms | Tokens: {result['tokens_generated']}")

In [ ]:
# Shut down the server
server_proc.terminate()
server_proc.wait()
print("Server stopped.")

## 4.3 Deploying to Google Cloud Run

We've verified the API works locally. Now let's deploy it as a **scalable cloud service**.

**The plan:**
1. **Here in Colab** - set up a GCP project and upload the trained checkpoint to Google Cloud Storage
2. **In Google Cloud Shell** - clone the repo, pull the checkpoint, build a Docker image, and deploy to Cloud Run

Cloud Run gives us: auto-scaling, HTTPS, load balancing, and pay-per-request pricing.

> **Note for attendees:** If you don't have a GCP account or can't set up billing, follow along with the instructor's demo.

In [ ]:
# --- GCP Project Setup (runs in Colab) ---
import subprocess

print("=" * 55)
print("  GCP PROJECT SETUP")
print("=" * 55)

# Step 1: Authenticate
print("\n[Step 1/4] Authenticating with Google Cloud...")
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("           Done.")
else:
    print("           Not in Colab - run 'gcloud auth login' in your terminal.")

# Step 2: Select or create a GCP project
print("\n[Step 2/4] Selecting GCP project...")
proj_list = subprocess.run(
    ["gcloud", "projects", "list", "--format", "value(projectId)"],
    capture_output=True, text=True
)
projects = [p.strip() for p in proj_list.stdout.strip().splitlines() if p.strip()]

if projects:
    print("           Your projects:")
    for i, p in enumerate(projects, 1):
        print(f"             {i}. {p}")
    print()

PROJECT_ID = ""
while not PROJECT_ID:
    PROJECT_ID = input("           Enter a GCP project ID: ").strip()
    if not PROJECT_ID:
        print("           Project ID is required. Try again.")

if PROJECT_ID not in projects:
    print(f"           Project '{PROJECT_ID}' not found - creating...")
    result = subprocess.run(
        ["gcloud", "projects", "create", PROJECT_ID, "--name", "PyTorch Workshop"],
        capture_output=True, text=True
    )
    if result.returncode != 0 and "already exists" not in result.stderr:
        print(f"           Error: {result.stderr.strip()}")
    else:
        print(f"           Created.")

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
               capture_output=True, check=True)
print(f"           Active project: {PROJECT_ID}")

# Step 3: Check billing - auto-link if possible (e.g. free trial users)
print("\n[Step 3/4] Checking billing...")
billing = subprocess.run(
    ["gcloud", "billing", "projects", "describe", PROJECT_ID,
     "--format", "value(billingAccountName)"],
    capture_output=True, text=True
)
billing_linked = bool(billing.stdout.strip())

if billing_linked:
    print(f"           Billing active: {billing.stdout.strip()}")
else:
    print("           No billing linked to this project.")
    accounts = subprocess.run(
        ["gcloud", "billing", "accounts", "list",
         "--format", "value(ACCOUNT_ID,DISPLAY_NAME)", "--filter", "open=true"],
        capture_output=True, text=True
    )
    available = [line.strip() for line in accounts.stdout.strip().splitlines() if line.strip()]

    acct_id = None
    if len(available) == 1:
        acct_id = available[0].split()[0]
        print(f"           Found billing account: {available[0]}")
        print(f"           Linking automatically...")
    elif len(available) > 1:
        print("           Available billing accounts:")
        for a in available:
            print(f" - {a}")
        acct_id = input("           Enter account ID to link: ").strip()

    if acct_id:
        link = subprocess.run(
            ["gcloud", "billing", "projects", "link", PROJECT_ID,
             f"--billing-account={acct_id}"],
            capture_output=True, text=True
        )
        if link.returncode == 0:
            print("           Billing linked successfully.")
            billing_linked = True
        else:
            print(f"           Failed: {link.stderr.strip()}")

    if not billing_linked:
        print("           Cloud Run requires billing. Link one at:")
        print(f"           https://console.cloud.google.com/billing/linkedaccount?project={PROJECT_ID}")
        print("           Then re-run this cell.")

# Step 4: Enable APIs
print("\n[Step 4/4] Enabling APIs (Cloud Build, Cloud Run, Storage)...")
if not billing_linked:
    print("           Skipped - link billing first, then re-run this cell.")
else:
    result = subprocess.run([
        "gcloud", "services", "enable",
        "cloudbuild.googleapis.com",
        "run.googleapis.com",
        "storage.googleapis.com",
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print("           Done.")
    else:
        print(f"           Error: {result.stderr.strip()}")

apis_ok = billing_linked and (result.returncode == 0 if billing_linked else False)

print("\n" + "=" * 55)
print(f"  Project:  {PROJECT_ID}")
print(f"  Billing:  {'Linked' if billing_linked else 'NOT LINKED'}")
print(f"  APIs:     {'Enabled' if apis_ok else 'Not enabled'}")
print(f"  Status:   {'Ready' if apis_ok else 'Fix above issues, then re-run'}")
print("=" * 55)
if apis_ok:
    print("\n  >>> Run the next cell to upload your model to GCS.")

### Upload your trained model to Google Cloud Storage

This copies your checkpoint and tokenizer to a GCS bucket so you can pull them in Cloud Shell.

In [ ]:
# --- Upload trained model to GCS ---
import subprocess
from pathlib import Path

BUCKET = f"gs://{PROJECT_ID}-workshop-artifacts"

# Create bucket
r = subprocess.run(
    ["gcloud", "storage", "buckets", "create", BUCKET, "--location=us-central1"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"Created bucket: {BUCKET}")
elif "already exists" in r.stderr or "409" in r.stderr:
    print(f"Bucket exists: {BUCKET}")
else:
    print(f"Bucket error: {r.stderr.strip()}")

# Find source files
tok = None
for p in ["../serve/tokenizer.json", "../tokenizer.json"]:
    if Path(p).exists():
        tok = p
        break

ckpt = None
for p in ["../serve/model_checkpoint.pt", "../checkpoints"]:
    if Path(p).is_file():
        ckpt = p
        break
    elif Path(p).is_dir():
        # Find latest checkpoint in the directory
        pts = sorted(Path(p).glob("checkpoint_*.pt"))
        if pts:
            ckpt = str(pts[-1])
            break

print(f"Tokenizer: {tok or 'NOT FOUND'}")
print(f"Checkpoint: {ckpt or 'NOT FOUND'}")

if not tok or not ckpt:
    print("\nSource files missing! Make sure you ran Notebooks 01-02 first")
    print("(or the training cell earlier in this notebook).")
    ok = False
else:
    ok = True
    for src, name in [(tok, "tokenizer.json"), (ckpt, "model_checkpoint.pt")]:
        r = subprocess.run(
            ["gcloud", "storage", "cp", src, f"{BUCKET}/{name}"],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            print(f"Uploaded {name} ({Path(src).stat().st_size / 1e6:.1f} MB)")
        else:
            print(f"FAILED to upload {name}: {r.stderr.strip()}")
            ok = False

if ok:
    print(f"\n" + "=" * 55)
    print(f"  DONE - model uploaded to {BUCKET}")
    print(f"=" * 55)
    print(f"\n  You're finished in Colab!")
    print(f"  Now open Google Cloud Shell and follow the steps below.")
    print(f"  https://shell.cloud.google.com")
    print(f"\n  First commands in Cloud Shell:")
    print(f"    gcloud config set project {PROJECT_ID}")
    print(f"    git clone https://github.com/arj7192/pytorch-production-workshop.git")
    print(f"    cd pytorch-production-workshop")
    print(f"\n  The Docker container pulls the model from GCS at startup - ")
    print(f"  no need to download model files manually.")
else:
    print(f"\n" + "=" * 55)
    print(f"  UPLOAD FAILED - check errors above.")
    print(f"=" * 55)
    print(f"  Common fixes:")
    print(f" - Ensure billing is linked to project '{PROJECT_ID}'")
    print(f" - Grant yourself Storage Admin: gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"        --member=user:$(gcloud config get account) --role=roles/storage.admin")
    print(f" - Then re-run this cell.")

In [ ]:
# --- Colab portion complete. The rest is a Cloud Shell walkthrough. ---

---

## 4.4 Switch to Google Cloud Shell

**Everything below runs in Cloud Shell, not Colab.**

Docker and `gcloud` can't run inside Colab, so we switch to [**Google Cloud Shell**](https://shell.cloud.google.com) - a free browser-based terminal with Docker and `gcloud` pre-installed.

Open it now: **[shell.cloud.google.com](https://shell.cloud.google.com)**

The rest of this notebook is a step-by-step guide. Copy-paste each command block into Cloud Shell.

### Step 1 - Set your project and clone the repo

```bash
# Set your project (replace with your project ID)
gcloud config set project YOUR_PROJECT_ID

# Clone the repo
git clone https://github.com/arj7192/pytorch-production-workshop.git
cd pytorch-production-workshop
```

No need to copy model files - the container pulls them from GCS automatically at startup.

### Step 2 - Build and test Docker locally

```bash
docker build -f serve/Dockerfile -t pytorch-workshop-api .
docker run -d -p 8080:8000 \
  -e GCS_BUCKET=gs://$GOOGLE_CLOUD_PROJECT-workshop-artifacts \
  pytorch-workshop-api
```

The container downloads `tokenizer.json` and `model_checkpoint.pt` from GCS on startup.

Test it:

```bash
curl http://localhost:8080/health

curl -X POST http://localhost:8080/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"Hello world","max_tokens":30}'
```

### Step 3 - Push image and deploy to Cloud Run

Push the image you just built to Google Container Registry:

```bash
gcloud auth configure-docker --quiet
docker tag pytorch-workshop-api gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api
docker push gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api
```

Deploy - **pick CPU or GPU:**

The container downloads model artifacts from GCS at startup (no model baked into the image).

**CPU** (default, no GPU quota needed):

```bash
gcloud run deploy pytorch-workshop-api \
  --image gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api \
  --platform managed --region us-central1 \
  --port 8000 --memory 2Gi --cpu 2 \
  --set-env-vars GCS_BUCKET=gs://$GOOGLE_CLOUD_PROJECT-workshop-artifacts \
  --allow-unauthenticated
```

**GPU** (requires GPU quota):

```bash
gcloud run deploy pytorch-workshop-api \
  --image gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api \
  --platform managed --region us-central1 \
  --port 8000 --memory 4Gi --cpu 4 --gpu 1 --gpu-type nvidia-l4 \
  --set-env-vars GCS_BUCKET=gs://$GOOGLE_CLOUD_PROJECT-workshop-artifacts \
  --allow-unauthenticated
```

> The same Docker image works for both - `serve/app.py` auto-detects GPU at startup and falls back to CPU.

**Or use the deploy script** to do steps 2-3 in one command:

```bash
chmod +x serve/deploy.sh
./serve/deploy.sh $GOOGLE_CLOUD_PROJECT          # CPU
./serve/deploy.sh $GOOGLE_CLOUD_PROJECT --gpu     # GPU
```

### Step 4 - Test your live endpoint

```bash
SERVICE_URL=$(gcloud run services describe pytorch-workshop-api \
  --region us-central1 --format "value(status.url)")

curl $SERVICE_URL/health

curl -X POST $SERVICE_URL/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"The future of AI","max_tokens":40}'
```

## 4.5 What to Monitor in Production (Reference)

Once deployed, you need to know when things go wrong. Key metrics:

| Metric | What it tells you | Alert threshold |
|--------|------------------|----------------|
| **Request latency (P99)** | User experience | >500ms for sync APIs |
| **Error rate** | Model/service health | >1% |
| **Memory usage** | OOM risk | >80% of limit |
| **Cold start time** | Time to first request | >10s |
| **Model staleness** | When was the model last updated | App-specific |

### Cloud Run gives you:
- Request count, latency, error rate (built-in)
- Container CPU/memory usage (built-in)
- Custom metrics via Cloud Logging (add to your app)

### Application-level logging (already in our API):
- Request ID for tracing
- Inference latency per request
- Input/output sizes

## Summary: What We Built Today

```
┌─────────────────────────────────────────────────────────┐
│                  Workshop Pipeline                      │
│                                                         │
│  ┌─────────┐   ┌──────────┐   ┌──────────┐   ┌──────┐ │
│  │ Module 1 │──▶│ Module 2 │──▶│ Module 3 │──▶│Mod. 4│ │
│  │ Train   │   │ Optimize │   │ Export   │   │Deploy│ │
│  └─────────┘   └──────────┘   └──────────┘   └──────┘ │
│                                                         │
│  Transformer     AMP           TorchScript    FastAPI   │
│  Training loop   Profiling     ONNX          Docker    │
│  Checkpoints     Stability     Quantization  Cloud Run │
│  Eval + logging  DataLoader    Benchmarking  Monitoring│
└─────────────────────────────────────────────────────────┘
```

You now have a complete, production-grade ML pipeline - from a raw model to a deployed, monitored inference service.

### Next steps for your own projects:
1. **Swap the model**: Replace `TransformerLM` with your model
2. **Scale up**: Add GPU Cloud Run instances for large models
3. **Add CI/CD**: GitHub Actions for automatic deployment
4. **A/B testing**: Deploy multiple model versions side by side
5. **Monitoring**: Add Prometheus metrics or Cloud Monitoring